# Q-learning - advanced walkthrough

Run every cell from top to bottom. The notebook prints intermediate values and draws visualizations so the math stays visible.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(7)
plt.style.use('default')

## 1. Define a tiny gridworld
The agent starts at the top-left and wants the goal at the bottom-right.

In [ ]:
rows, cols = 3, 3
start = (0, 0)
goal = (2, 2)
actions = ['up', 'down', 'left', 'right']
deltas = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}

def step(state, action):
    if state == goal:
        return state, 0, True
    dr, dc = deltas[action]
    nr = min(max(state[0] + dr, 0), rows - 1)
    nc = min(max(state[1] + dc, 0), cols - 1)
    next_state = (nr, nc)
    reward = 1 if next_state == goal else -0.04
    done = next_state == goal
    return next_state, reward, done

print('States:', [(r, c) for r in range(rows) for c in range(cols)])
print('Actions:', actions)

## 2. Show one Q-learning update

In [ ]:
Q = {(r, c): {a: 0.0 for a in actions} for r in range(rows) for c in range(cols)}
state = (2, 1)
action = 'right'
alpha = 0.5
gamma = 0.9
next_state, reward, done = step(state, action)
old = Q[state][action]
target = reward + gamma * max(Q[next_state].values())
new = old + alpha * (target - old)
print('state:', state)
print('action:', action)
print('next_state:', next_state)
print('reward:', reward)
print('old Q:', old)
print('target:', target)
print('new Q:', new)
Q[state][action] = new

## 3. Train with epsilon-greedy exploration

In [ ]:
Q = {(r, c): {a: 0.0 for a in actions} for r in range(rows) for c in range(cols)}
alpha = 0.4
gamma = 0.95
epsilon = 0.2
episode_returns = []

for episode in range(300):
    state = start
    total_reward = 0
    for t in range(40):
        if np.random.rand() < epsilon:
            action = np.random.choice(actions)
        else:
            action = max(Q[state], key=Q[state].get)
        next_state, reward, done = step(state, action)
        target = reward + gamma * max(Q[next_state].values())
        Q[state][action] += alpha * (target - Q[state][action])
        state = next_state
        total_reward += reward
        if done:
            break
    episode_returns.append(total_reward)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(pd.Series(episode_returns).rolling(20).mean())
ax.set_title('Q-learning return improves over episodes')
ax.set_xlabel('episode')
ax.set_ylabel('rolling mean return')
plt.show()

## 4. Visualize learned state values and policy

In [ ]:
V = np.zeros((rows, cols))
policy = np.empty((rows, cols), dtype=object)
arrow = {'up': '^', 'down': 'v', 'left': '<', 'right': '>'}
for r in range(rows):
    for c in range(cols):
        s = (r, c)
        best_action = max(Q[s], key=Q[s].get)
        V[r, c] = max(Q[s].values())
        policy[r, c] = 'G' if s == goal else arrow[best_action]

print('Policy arrows:')
display(pd.DataFrame(policy))
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(V, cmap='YlGnBu')
for r in range(rows):
    for c in range(cols):
        ax.text(c, r, policy[r, c], ha='center', va='center', fontsize=16)
ax.set_title('Learned value heatmap and greedy policy')
ax.set_xticks(range(cols))
ax.set_yticks(range(rows))
fig.colorbar(im, ax=ax)
plt.show()

Try setting `epsilon = 0.0`. Without exploration, the agent may fail to discover useful actions.